# Chapter 12: Serial Correlation and Heteroscedasticity in Time Series

Topics: testing for serial correlation, FGLS, HAC standard errors, ARCH/GARCH.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_arch
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
df = woo.dataWoo('phillips')
model = smf.ols('inf ~ unem', data=df).fit()

## 12.1 Testing for Serial Correlation

In [ ]:
# Durbin-Watson
dw = durbin_watson(model.resid)
print(f"Durbin-Watson: {dw:.4f}")
print("DW ≈ 2: no autocorrelation | DW < 2: positive | DW > 2: negative")

# Breusch-Godfrey test
bg_stat, bg_pval, _, _ = acorr_breusch_godfrey(model, nlags=2)
print(f"\nBreusch-Godfrey (2 lags): stat={bg_stat:.4f}, p-value={bg_pval:.4f}")
print("→ Reject H0 of no serial correlation" if bg_pval < 0.05 else "→ Fail to reject H0")

## ACF and PACF of Residuals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(model.resid,  ax=axes[0], lags=20, title='ACF of Residuals')
plot_pacf(model.resid, ax=axes[1], lags=20, title='PACF of Residuals')
plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_acf_pacf.png", dpi=150)
plt.show()

## 12.2 FGLS with AR(1) Errors

In [ ]:
from statsmodels.regression.linear_model import GLSAR
import statsmodels.api as sm

y = df['inf'].values
X = sm.add_constant(df['unem'].values)

gls = GLSAR(y, X, rho=1)
gls_result = gls.iterative_fit(maxiter=10)
print("FGLS (Cochrane-Orcutt) Results:")
print(gls_result.summary())

## 12.3 HAC Standard Errors

In [ ]:
m_hac = smf.ols('inf ~ unem', data=df).fit(cov_type='HAC', cov_kwds={'maxlags': 2})
print("OLS with Newey-West HAC Standard Errors:")
print(m_hac.summary())

## 12.4 ARCH Test

In [ ]:
arch_stat, arch_pval, _, _ = het_arch(model.resid, nlags=4)
print(f"ARCH test (4 lags): stat={arch_stat:.4f}, p-value={arch_pval:.4f}")
print("→ Evidence of ARCH effects" if arch_pval < 0.05 else "→ No ARCH effects")